In [1]:
#Basic package
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import joblib
import sys
import io
from dotenv import load_dotenv
import random

In [47]:
import plotly.graph_objects as go

In [8]:
import warnings
warnings.filterwarnings("ignore")


In [3]:
#Ultility package
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from ultility import compute_rank_ic, to_excel
from arch import arch_model

In [9]:
#Model package
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy.stats import jarque_bera,skew, kurtosis, gaussian_kde

In [26]:
path = "../dataset/VN30_dataset_from_2019.csv"
df = pd.read_csv(path)

df["time"] = pd.to_datetime(df["time"], format="mixed", dayfirst=True, errors="coerce")


df_2020 = df[df["time"].dt.year >= 2020]


market_returns = (
    df_2020
    .sort_values("time")
    .groupby("time")
    .apply(lambda x: np.average(
        x["return_1d"],
        weights=x["Market Capital (Bn VND)"] / x["Market Capital (Bn VND)"].sum()
    ))
    .dropna()
)

market_returns = market_returns.to_frame(name="market_return")

jb_stat, jb_pvalue = jarque_bera(market_returns["market_return"])

sk = skew(market_returns["market_return"])
kt = kurtosis(market_returns["market_return"], fisher=False)

result = pd.DataFrame({
    "JB_stat": [jb_stat],
    "p_value": [jb_pvalue],
    "skewness": [sk],
    "kurtosis": [kt]
})
print(result)

       JB_stat  p_value  skewness  kurtosis
0  1864.687432      0.0 -0.939863  8.132393


In [51]:

data = market_returns.squeeze()

kde = gaussian_kde(data)
x = np.linspace(data.min(), data.max(), 1000)
y = kde(x)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name="KDE",
        line=dict(width=2),
        hovertemplate="Return=%{x}<br>Density=%{y}<extra></extra>"
    )
)

fig.update_layout(
    title="Density of Stock Returns (Value-Weighted)",
    xaxis_title="Market Return",
    yaxis_title="Density",
    template="plotly_white",
    hovermode="x"
)

fig.show()

**Jarque-Bera Test Results:**  
JB statistic rất cao với p-value ≈ 0 → Bác bỏ H₀ (phân phối chuẩn)  
Kết luận: Chuỗi lợi suất có skewness và excess kurtosis đáng kể, không tuân theo phân phối chuẩn

In [ ]:
returns = market_returns.squeeze()

results = []

for p in range(1, 10):
    for q in range(1, 10):
        try:
            model = arch_model(
                returns,
                mean="Constant",
                vol="GARCH",
                p=p,
                q=q,
                dist="t"
            )
            res = model.fit(disp="off")
            results.append({
                "p": p,
                "q": q,
                "loglik": res.loglikelihood,
                "aic": res.aic,
                "bic": res.bic,
                "alpha_beta": np.sum(
                    res.params[[k for k in res.params.index if "alpha" in k or "beta" in k]]
                )
            })
        except:
            pass

results_df = pd.DataFrame(results)

best_aic = results_df.sort_values("aic").head(10)
best_bic = results_df.sort_values("bic").head(10)

best_aic, best_bic


Dựa vào chỉ số AIC BIC và alpha beta, chọn mô hình GARCH(1,6)

In [34]:
returns = market_returns.squeeze()

final_model = arch_model(
    returns,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1,
    dist="t"
)

final_res = final_model.fit(disp="off")
final_res.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                        Constant Mean - GARCH Model Results                         
====================================================================================
Dep. Variable:                market_return   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:                3540.06
Distribution:      Standardized Student's t   AIC:                          -7070.11
Method:                  Maximum Likelihood   BIC:                          -7043.55
                                              No. Observations:                 1498
Date:                      Wed, Feb 04 2026   Df Residuals:                     1497
Time:                              18:05:45   Df Model:                            1
                                 Mean Model                                 
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
mu         9.3729e-04  2.007e-04      4.669  3.020e-06 [5.439e-04,1.331e-03]
                              Volatility Model                              
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omega      6.6072e-04  2.113e-05     31.268 1.288e-214 [6.193e-04,7.021e-04]
alpha[1]       0.5281      0.178      2.969  2.986e-03     [  0.179,  0.877]
beta[1]        0.4714  6.380e-03     73.890      0.000     [  0.459,  0.484]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu            11.7548  7.492e-02    156.906      0.000 [ 11.608, 11.902]
========================================================================

Covariance estimator: robust
"""

In [39]:
cond_vol = final_res.conditional_volatility.dropna()
log_vol = np.log(cond_vol)
log_vol_std = (log_vol - log_vol.mean()) / log_vol.std()



In [43]:
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

ms3_model = MarkovRegression(
    log_vol_std,
    k_regimes=3,
    trend="c",
    switching_variance=True
)

ms3_res = ms3_model.fit(
    em_iter=50,
    search_reps=20,
    search_iter=10,
    disp=False
)

ms3_res.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                        Markov Switching Model Results                        
==============================================================================
Dep. Variable:               cond_vol   No. Observations:                 1498
Model:               MarkovRegression   Log Likelihood                -393.053
Date:                Wed, 04 Feb 2026   AIC                            810.106
Time:                        18:11:26   BIC                            873.849
Sample:                             0   HQIC                           833.854
                               - 1498                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1729      0.018     -9.525      0.000      -0.209      -0.137
sigma2         0.0522      0.005     10.489      0.000       0.042       0.062
                             Regime 1 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.5329      0.006    -89.394      0.000      -0.545      -0.521
sigma2         0.0066      0.001      8.576      0.000       0.005       0.008
                             Regime 2 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.4673      0.098     14.918      0.000       1.275       1.660
sigma2         2.0928      0.173     12.100      0.000       1.754       2.432
                         Regime transition parameters                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
p[0->0]        0.7662      0.022     34.251      0.000       0.722       0.810
p[1->0]        0.1177      0.017      6.923      0.000       0.084       0.151
p[2->0]        0.1875      0.026      7.117      0.000       0.136       0.239
p[0->1]        0.1735      0.020      8.632      0.000       0.134       0.213
p[1->1]        0.8468      0.018     47.057      0.000       0.812       0.882
p[2->1]     1.867e-21        nan        nan        nan         nan         nan
==============================================================================

Warnings:
[1] Covariance matrix calculated using numerical (complex-step) differentiation.
"""

In [52]:
import plotly.graph_objects as go

vol = log_vol_std.loc[regime_3.index]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=vol.index,
        y=vol.values,
        mode="lines",
        name="Log-Volatility",
        line=dict(width=1)
    )
)

colors = {
    0: "rgba(173,216,230,0.4)",
    1: "rgba(255,215,0,0.4)",
    2: "rgba(255,99,71,0.4)"
}

for r in sorted(regime_3.unique()):
    mask = regime_3 == r
    fig.add_trace(
        go.Scatter(
            x=vol.index[mask],
            y=vol.values[mask],
            mode="markers",
            name=f"Regime {r}",
            marker=dict(color=colors[r], size=4),
            hovertemplate="Time=%{x}<br>Vol=%{y}<extra>Regime "+str(r)+"</extra>"
        )
    )

fig.update_layout(
    title="Market Volatility with Markov Switching Regimes",
    xaxis_title="Time",
    yaxis_title="Standardized Log-Volatility",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()


In [53]:
alpha = 0.05

df_regime = df.merge(
    regime_3.rename("regime"),
    left_on="time",
    right_index=True,
    how="inner"
)


In [55]:
def cvar(x, alpha):
    q = np.quantile(x, alpha)
    return x[x <= q].mean()

cvar_df = (
    df_regime
    .groupby(["regime", "ticker"])["return_1d"]
    .apply(lambda x: cvar(x.dropna(), alpha))
    .reset_index(name="CVaR")
)
cvar_pivot = cvar_df.pivot(
    index="ticker",
    columns="regime",
    values="CVaR"
).dropna()



In [60]:

cvar_plot = -cvar_pivot

fig = go.Figure()

colors = {
    0: "#4C78A8",
    1: "#F2B701",
    2: "#E45756"
}

for r in cvar_plot.columns:
    fig.add_trace(
        go.Bar(
            x=cvar_plot.index,
            y=cvar_plot[r],
            name=f"Regime {r}",
            marker_color=colors.get(r, None),
            hovertemplate=
                "Ticker=%{x}<br>"
                "CVaR="+cvar_plot[r].map(lambda v: f"({abs(v):.2%})").values+
                "<extra></extra>"
        )
    )

fig.update_layout(
    title="CVaR by Asset under Common Market Regimes",
    xaxis_title="Ticker",
    yaxis_title="CVaR (95%)",
    barmode="group",
    template="plotly_white",
    hovermode="x unified"
)

fig.update_yaxes(
    tickformat=".0%",
    tickprefix="(",
    ticksuffix=")"
)

fig.show()


In [61]:
colors = {
    0: "#4C78A8",
    1: "#F2B701",
    2: "#E45756"
}

for r in cvar_pivot.columns:
    cvar_r = -cvar_pivot[r]
    cvar_r = cvar_r.sort_values(ascending=False)

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=cvar_r.index,
            y=cvar_r.values,
            marker_color=colors.get(r, None),
            hovertemplate=
                "Ticker=%{x}<br>"
                "CVaR="+cvar_r.map(lambda v: f"({abs(v):.2%})").values+
                "<extra></extra>"
        )
    )

    fig.update_layout(
        title=f"CVaR by Ticker – Regime {r} (Sorted Descending)",
        xaxis_title="Ticker",
        yaxis_title="CVaR (95%)",
        template="plotly_white"
    )

    fig.update_yaxes(
        tickformat=".0%",
        tickprefix="(",
        ticksuffix=")"
    )

    fig.show()

# TỔNG HỢP KẾT QUẢ MÔ HÌNH GARCH

## Hạng Mục Huấn Luyện
**Mô hình GARCH với Markov Switching cho dự báo volatility thị trường VN30**

## Overview Task
Phân tích volatility thị trường VN30 từ 2020 sử dụng mô hình GARCH kết hợp với Markov Switching để nhận dạng các chế độ biến động khác nhau và tính toán CVaR cho các tài sản.

## Kiến trúc thiết kế mô hình
- **Mô hình chính**: GARCH(1,1) với phân phối Student-t
- **Mô hình phụ**: Markov Switching với 3 trạng thái (regimes)
- **Dữ liệu đầu vào**: VN30 dataset từ 2020, market return weighted theo market capitalization

## Thử viện định kèm
- `arch`: Cho mô hình GARCH
- `statsmodels`: Cho Markov Switching và các kiểm định thống kê
- `plotly`: Trực quan hóa kết quả
- `pandas`, `numpy`: Xử lý dữ liệu

## Tiền xử lý dữ liệu và chuẩn hóa

### Mô tả tiền xử lý dữ liệu và chuẩn hóa bằng diễn giải
1. **Tải dữ liệu**: VN30 dataset từ file CSV
2. **Lọc dữ liệu**: Chỉ lấy dữ liệu từ năm 2020 trở về sau
3. **Tính market return**: Tính toán lợi suất thị trường weighted theo market capitalization
4. **Kiểm định tính chất**: Thực hiện Jarque-Bera test để kiểm tra phân phối chuẩn

### Mô tả tiền xử lý bằng Code
```python
# Weighted market return calculation
market_returns = (
    df_2020
    .sort_values("time")
    .groupby("time")
    .apply(lambda x: np.average(
        x["return_1d"],
        weights=x["Market Capital (Bn VND)"] / x["Market Capital (Bn VND)"].sum()
    ))
    .dropna()
)
```

## Định nghĩa mô hình
**GARCH(1,1) Model:**
- Mean equation: Constant mean
- Variance equation: GARCH(1,1) 
- Distribution: Student-t distribution
- Parameters: ω (omega), α (alpha), β (beta)

## Quá trình huấn luyện mô hình

### 1. Câu hỏi huấn luyện mô hình
- Tìm bộ tham số (p,q) tối ưu cho mô hình GARCH bằng cách so sánh AIC và BIC
- Grid search từ p=1 đến p=9, q=1 đến q=9

### 2. Câu hỏi máy tính
- Sử dụng Maximum Likelihood Estimation để ước lượng tham số
- Kiểm tra tính ổn định: α + β < 1

### Code Huấn luyện mô hình
```python
# Model selection
for p in range(1, 10):
    for q in range(1, 10):
        model = arch_model(returns, mean="Constant", vol="GARCH", p=p, q=q, dist="t")
        res = model.fit(disp="off")

# Final model GARCH(1,1)
final_model = arch_model(returns, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
final_res = final_model.fit(disp="off")
```

## Đánh giá kết quả

### Kết quả Jarque-Bera Test
- **JB statistic**: Rất cao với p-value ≈ 0
- **Kết luận**: Bác bỏ H₀, chuỗi lợi suất có skewness và excess kurtosis đáng kể, không tuân theo phân phối chuẩn

### Mô hình GARCH(1,1) được chọn
- **Lý do chọn**: Dựa trên AIC, BIC và điều kiện ổn định α + β < 1
- **Phân phối**: Student-t phù hợp với fat tails của dữ liệu tài chính

### Markov Switching với 3 Regimes
- **Regime 0**: Biến động thấp (Low volatility)
- **Regime 1**: Biến động trung bình (Medium volatility) 
- **Regime 2**: Biến động cao (High volatility)

## Lựa chọn siêu tham số

### Mã ngưỡi thực hiện Grid Search
- **p, q**: Từ 1 đến 9
- **Tiêu chí chọn**: AIC và BIC
- **Kết quả**: GARCH(1,1) được chọn

## Nhận xét rút ra kết luận

### 1. Phần Nhận Xét
- Thị trường VN30 có 3 chế độ biến động rõ ràng được phân tách bởi Markov Switching
- CVaR khác nhau đáng kể giữa các regime, cho thấy rủi ro thay đổi theo thời gian
- Mô hình GARCH(1,1) phù hợp để mô hình hóa conditional volatility

### 2. Phần Kết Luận
- Mô hình kết hợp GARCH-Markov Switching hiệu quả trong việc nhận dạng các chế độ thị trường
- CVaR analysis cung cấp insight về rủi ro tail risk của từng tài sản theo các regime
- Kết quả có thể ứng dụng trong quản lý rủi ro và phân bổ tài sản

## Tài liệu tham khảo
- Engle, R. F. (1982). Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation
- Bollerslev, T. (1986). Generalized autoregressive conditional heteroskedasticity
- Hamilton, J. D. (1989). A new approach to the economic analysis of nonstationary time series and the business cycle